In [1]:
import polars as pl
import re
import os
import sys
from collections import Counter
from concurrent.futures import ProcessPoolExecutor,as_completed
from multiprocessing import Lock
import pyarrow.parquet as pq
import pyarrow as pa
from typing import List
import pandas as pd
import argparse
import json
import gzip
from datetime import datetime
import gc
from natsort import natsorted
from Bio import SeqIO,AlignIO
import subprocess
from io import StringIO
import pysam
import tempfile
import math
import csv
import io
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
import numpy as np

In [ ]:
def view(df):
    import pandas as pd
    from pandas import DataFrame
    df_pandas = df.to_pandas()
    return df_pandas

def getFixedSites(code_parquet, sample_list, group_name):
    fixed_set = {1,2,3,4,16,33,34,35,36,48}

    lazy = pl.scan_parquet(code_parquet).select(
        ['contig_index', 'contig_position'] + sample_list
    )

    all_missing = (
        lazy
        .filter(pl.all_horizontal([pl.col(s) == 0 for s in sample_list]))
        .with_columns([
            pl.lit(0).cast(pl.Int32).alias(group_name),
            pl.lit(len(sample_list)).cast(pl.Int32).alias('Fixed_Count')
        ])
        .select([
            "contig_index",
            "contig_position",
            group_name,
            "Fixed_Count"
        ])
    )

    fixed = (
        lazy
        .filter(~pl.all_horizontal([pl.col(s) == 0 for s in sample_list]))
        .filter(
            pl.max_horizontal([
                pl.when(pl.col(s) != 0).then(pl.col(s)).otherwise(None) 
                for s in sample_list
            ]) ==
            pl.min_horizontal([
                pl.when(pl.col(s) != 0).then(pl.col(s)).otherwise(None) 
                for s in sample_list
            ])
        )
        .with_columns([
            pl.max_horizontal([pl.col(s) for s in sample_list]).cast(pl.Int32).alias(group_name)
        ])
        .filter(pl.col(group_name).is_in(fixed_set))
        .with_columns([
            pl.sum_horizontal([(pl.col(s) != 0).cast(pl.Int32) for s in sample_list]).alias("Fixed_Count")
        ])
        .select(['contig_index','contig_position', group_name, 'Fixed_Count'])
    )


    result = (
        pl.concat([all_missing, fixed])
        .select([
            "contig_index",
            "contig_position",
            group_name,
            "Fixed_Count"
        ])
        .sort(['contig_index','contig_position'])
        .collect()
    )

    return result

def snpSubtractor(fixed_df, focal_id, code_file, subtract_samples):

    fixed_codes = {1, 2, 3, 4, 16}

    degenerate_map = {
        1: {5,21,6,22,7,23,11,27,12,28,13,29,15,17,31},  # A
        2: {5,21,8,24,9,25,11,27,12,28,14,30,15,18,31},  # C
        3: {6,22,8,24,10,26,11,27,13,29,14,30,15,19,31}, # G
        4: {7,23,9,25,10,26,12,28,13,29,14,30,15,20,31}, # T
        16: {17,18,29,20,21,22,23,24,25,26,27,28,29,30,31},  # GAP
    }

    focal_df = (
        fixed_df
        .filter(pl.col(focal_id) > 0)
        .with_columns(
            pl.when(pl.col(focal_id) >= 33)
            .then(pl.col(focal_id) - 32)
            .otherwise(pl.col(focal_id))
            .alias(focal_id)
        )
    )

    site_count = focal_df.height
    lazy_code = pl.scan_parquet(code_file)

    sample_rows = []
    for sample in subtract_samples:

        if site_count == 0:
            print("No sites remain!")
            break

        print(f"Removing sites where {sample} matches {focal_id}... Starting count {site_count}.")

        sample_df = (
            lazy_code
            .select(['contig_index', 'contig_position', sample])
            .with_columns(
                pl.when(pl.col(sample) >= 33)
                .then(pl.col(sample) - 32)
                .otherwise(pl.col(sample))
                .alias(sample)
            )
            .collect()
        )

        compare_df = focal_df.join(
            sample_df, on=['contig_index','contig_position'], how="left"
        )

        fixed_sample_df = compare_df.filter(pl.col(sample).is_in(fixed_codes)).with_columns(
            (pl.col(focal_id) == pl.col(sample)).alias("Match")
        )

        het_sample_df = (
            compare_df
            .filter(~pl.col(sample).is_in(fixed_codes))
            .with_columns([
                (
                    pl.when(pl.col(focal_id) == 1).then(pl.col(sample).is_in(degenerate_map[1]))
                    .when(pl.col(focal_id) == 2).then(pl.col(sample).is_in(degenerate_map[2]))
                    .when(pl.col(focal_id) == 3).then(pl.col(sample).is_in(degenerate_map[3]))
                    .when(pl.col(focal_id) == 4).then(pl.col(sample).is_in(degenerate_map[4]))
                    .when(pl.col(focal_id) == 16).then(pl.col(sample).is_in(degenerate_map[16]))
                    .otherwise(False)
                    .alias("Match")
                )
            ])
        )

        ploidy_fail_df = compare_df.filter(pl.col(sample) < 0)

        fixed_count = fixed_sample_df.height
        fixed_match_count = fixed_sample_df.filter(pl.col("Match")).height

        het_count = het_sample_df.height
        het_match_count = het_sample_df.filter(pl.col("Match")).height

        ploidy_fail_count = ploidy_fail_df.height

        total_covered = fixed_count + het_count + ploidy_fail_count

        print(f"{sample}: {total_covered} sites covered")
        print(f"  Fixed      : {fixed_match_count}/{fixed_count}")
        print(f"  Heterozygous: {het_match_count}/{het_count}")
        print(f"  Ploidy fails: {ploidy_fail_count}")

        sample_rows.append([sample,total_covered,fixed_count,fixed_match_count,het_count,het_match_count,ploidy_fail_count])

        match_df = pl.concat([
            fixed_sample_df.filter(pl.col("Match")).select(['contig_index','contig_position']),
            het_sample_df.filter(pl.col("Match")).select(['contig_index','contig_position']),
            ploidy_fail_df.select(['contig_index','contig_position'])
        ])

        focal_df = focal_df.join(
            match_df, on=['contig_index','contig_position'], how="anti"
        )

        site_count = focal_df.height

    site_removal_df = pl.DataFrame(
        sample_rows,
        schema=[
            "Sample_ID",
            "Covered",
            "Fixed",
            "Fixed_Match",
            "Heterozygous",
            "Het_Match",
            "Ploidy_Fail"
        ]
    )

    return focal_df, site_removal_df

def snpClassifier(snp_df, sample_id, called_base_path):

    fixed_codes = {1, 2, 3, 4, 16}

    degenerate_map = {
        1: {5,21,6,22,7,23,11,27,12,28,13,29,15,17,31},  # A
        2: {5,21,8,24,9,25,11,27,12,28,14,30,15,18,31},  # C
        3: {6,22,8,24,10,26,11,27,13,29,14,30,15,19,31}, # G
        4: {7,23,9,25,10,26,12,28,13,29,14,30,15,20,31}, # T
        16: {17,18,29,20,21,22,23,24,25,26,27,28,29,30,31},  # GAP
    }

    groups = snp_df["Focal_Group"].unique().to_list()
    sample_called_bases = pl.read_parquet(called_base_path)

    sample_rows = []

    for gr in groups:
        gr_snp_df = snp_df.filter((pl.col("Focal_Group")==gr))

        gr_class_df = gr_snp_df.join(sample_called_bases,on=['contig_index','contig_position'],how="left").filter(~(pl.col("base_code").is_null()))
    
        fixed_sample_df = gr_class_df.filter(pl.col("base_code").is_in(fixed_codes)).with_columns(
                (pl.col("Focal_Base") == pl.col("base_code")).alias("Match")
            )
    
        het_sample_df = (
            gr_class_df
            .filter(~pl.col("base_code").is_in(fixed_codes))
            .with_columns([
                (
                    pl.when(pl.col("Focal_Base") == 1).then(pl.col("base_code").is_in(degenerate_map[1]))
                    .when(pl.col("Focal_Base") == 2).then(pl.col("base_code").is_in(degenerate_map[2]))
                    .when(pl.col("Focal_Base") == 3).then(pl.col("base_code").is_in(degenerate_map[3]))
                    .when(pl.col("Focal_Base") == 4).then(pl.col("base_code").is_in(degenerate_map[4]))
                    .when(pl.col("Focal_Base") == 16).then(pl.col("base_code").is_in(degenerate_map[16]))
                    .otherwise(False)
                    .alias("Match")
                    )
                ])
            )
        
        ploidy_fail_df = gr_class_df.filter(pl.col("base_code") < 0)

        fixed_count = fixed_sample_df.height
        fixed_match_count = fixed_sample_df.filter(pl.col("Match")).height

        het_count = het_sample_df.height
        het_match_count = het_sample_df.filter(pl.col("Match")).height

        ploidy_fail_count = ploidy_fail_df.height

        total_covered = fixed_count + het_count + ploidy_fail_count

        sample_rows.append([sample_id,gr,total_covered,fixed_count,fixed_match_count,het_count,het_match_count,ploidy_fail_count])

    sample_df = pl.DataFrame(
        sample_rows,
        schema=[
            "Sample_ID",
            "Focal_Group",
            "Covered",
            "Fixed",
            "Fixed_Match",
            "Heterozygous",
            "Het_Match",
            "Ploidy_Fail"
        ],
        orient="row"
    )

    return sample_df

def rawSNPClassifer(snp_df, sample_id, raw_parquet_path):

    convert_dict = {1:"A",2:"C",3:"G",4:"T",16:"-"}

    fixed_codes = {1, 2, 3, 4, 16}

    degenerate_map = {
        1: {5,21,6,22,7,23,11,27,12,28,13,29,15,17,31},  # A
        2: {5,21,8,24,9,25,11,27,12,28,14,30,15,18,31},  # C
        3: {6,22,8,24,10,26,11,27,13,29,14,30,15,19,31}, # G
        4: {7,23,9,25,10,26,12,28,13,29,14,30,15,20,31}, # T
        16: {17,18,29,20,21,22,23,24,25,26,27,28,29,30,31},  # GAP
    }

    groups = snp_df["Focal_Group"].unique().to_list()
    sample_parquet = pl.read_parquet(raw_parquet_path)

    sample_rows = []

    for gr in groups:
        
        gr_snp_df = snp_df.filter((pl.col("Focal_Group")==gr))

        match_df = (
            gr_snp_df.join(sample_parquet, on=['contig_index','contig_position'], how="left")
            .select(['contig_index','contig_position','Focal_Base','base','frequency','depth'])
            .filter(~pl.col('depth').is_null())
            .with_columns([
                pl.col('Focal_Base').replace_strict(convert_dict).alias('Focal_Base')
            ])
            .with_columns([
                (pl.col('Focal_Base') == pl.col("base")).alias("Match")
            ])
            .with_columns([
                (pl.col('frequency') * pl.col('depth')).alias("allele_depth")
            ])
        )

        distinct_count = (
            match_df
            .group_by(["contig_index", "contig_position"])
            .len()
            .height
        )

        match_summary_df = (
            match_df
            .group_by(['Match'])
            .agg([
                pl.col("allele_depth").sum().alias("Sum_Allele_Depth")
            ])
        )

        count_match_df = match_summary_df.filter(pl.col("Match") == True)
        count_nonmatch_df = match_summary_df.filter(pl.col("Match") == False)

        if count_match_df.height == 0:
            sum_match = 0
        else:
            sum_match = count_match_df.select("Sum_Allele_Depth").item()

        if count_nonmatch_df.height == 0:
            sum_nonmatch = 0
        else:
            sum_nonmatch = count_nonmatch_df.select("Sum_Allele_Depth").item()

        sample_rows.append([sample_id,gr,distinct_count,sum_match, sum_nonmatch])

    sample_df = pl.DataFrame(
        sample_rows,
        schema=[
            "Sample_ID",
            "Focal_Group",
            "SNP_Count",
            "Match",
            "Non_Match"
        ],
        orient="row"
    ).with_columns([
    pl.col("SNP_Count").cast(pl.Int32),
    pl.col("Match").cast(pl.Int32),
    pl.col("Non_Match").cast(pl.Int32),
])

    return sample_df

In [ ]:
# Fetch data from JSON file
json_file = ""

with open(json_file, "r") as f:
    data = json.load(f)

join_id = data["Join_ID"]
joined_directory = data["Joined_Directory"]
sample_ids = natsorted(data["Sample_IDs"].split(","))
scaffold_file = data["Scaffold_File"]
code_file = data["Code_File"]
site_file = data["Site_File"]
sample_summary_file = data["Sample_Summary_File"]
site_count_file = data["Site_Count_File"]

lazy_codes = pl.scan_parquet(code_file)
lazy_sites = pl.scan_parquet(site_file)

In [3]:
# Create Alignment

tree_sites = (
    lazy_sites
    .filter(pl.col("Nonsingleton_Alleles") >= 2)
    .filter(pl.col("Hets") == 0)
    .filter(pl.col("Filtered") == 0)
    .filter(pl.col("Missing") <= 0)
    .select(["contig_index", "contig_position", "Missing"])
)

base_convert_dict = { 0:'-',
                     1:'A',2:"C",3:"G",4:"T",
                     33:'A', 34:'C', 35:'T', 36:'G'}
base_set = {0,1,2,3,4,33,34,35,36}

allowed = pl.lit(list(base_set))

lazy_tree_base_codes = (
    tree_sites
    .join(lazy_codes, on=["contig_index","contig_position"], how="left")
    .filter(
        pl.fold(
            acc=True,
            exprs=[pl.col(s).is_in(allowed) for s in sample_ids],
            function=lambda acc, x: acc & x
        )
    )
).collect(engine="streaming")

base_convert_dict = { 0:'-',
                     1:'A',2:"C",3:"G",4:"T",
                     33:'A', 34:'C', 35:'T', 36:'G'}

base_set = {0,1,2,3,4,33,34,35,36}

base_merged_df = lazy_tree_base_codes.with_columns([
    pl.col(col)
      .replace_strict(base_convert_dict)   
      .alias(col)
    for col in sample_ids
])

base_seq_records = []
for sample in sample_ids:
    sequence = "".join(base_merged_df[sample].to_list()) 
    seq_record = SeqRecord(Seq(sequence), id=sample)
    base_seq_records.append(seq_record)

base_alignment = MultipleSeqAlignment(base_seq_records)

base_alignment_file = ""
AlignIO.write(base_alignment, base_alignment_file, "fasta")

In [ ]:
# Set sample groups

data_dir = ""
group_df = pd.read_csv(os.path.join(data_dir,'group_data.tsv'),sep="\t")

group_data =  (
    group_df.groupby("Group")["ID"]
    .apply(list)
    .to_dict()
)

all_ids = list({id for ids in group_data.values() for id in ids})

In [ ]:
# Get fixed sites for each group
fixed_sites_dir = ""

fixed_sites = [
    getFixedSites(code_file, ids, group)
    for group, ids in group_data.items()
]

for i,(group, ids) in enumerate(group_data.items()):
    
    out_path = os.path.join(fixed_sites_dir, f"{group}.parquet")
    fixed_sites[i].write_parquet(out_path,compression="snappy")

In [ ]:
# Get group SNPs
snp_dir = ""

snp_results = []
site_results = []

for i,(group, ids) in enumerate(group_data.items()):

    snp_path = os.path.join(snp_dir, f"{group}.parquet")
    #site_path = os.path.join(snp_dir, f"{group}_sites.tsv")

    non_focal = list(set(all_ids) - set(ids))
    sp_snp_df,sp_site_df = snpSubtractor(fixed_sites[i],group,code_file,non_focal)

    snp_results.append(sp_snp_df.with_columns(pl.lit(group).alias("Focal_Group"),
    pl.col(group).alias("Focal_Base")).select(['contig_index','contig_position','Focal_Group','Focal_Base','Fixed_Count']))

    site_results.append(sp_site_df)

snp_df = pl.concat(snp_results)

snp_df.write_parquet(os.path.join(snp_dir,"SNPs.parquet"),compression="snappy")


In [ ]:
# Classify via called bases

called_results = []
called_base_paths = []

for called_path in called_base_paths:

    sample_id = os.path.basename(called_path).replace("_Called.parquet","")
    called_results.append(snpClassifier(snp_df,sample_id,called_path))

called_class_results = pl.concat(called_results)

In [ ]:
# Classify via raw parquets

raw_results = []
raw_parquet_paths = []

for raw_parquet in raw_parquet_paths:
    sample_id = os.path.basename(raw_parquet).replace("_Raw.parquet","")
    raw_results.append(rawSNPClassifer(snp_df,sample_id,raw_parquet))

raw_class_results = pl.concat(raw_results)